In [1]:
# ============================================================
# 0. Setup and raw data loading
# ============================================================

from pathlib import Path
import sys
import platform
import numpy as np
import pandas as pd

RANDOM_STATE = 9890
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

# Change this only if your CSV files are stored in another folder.
DATA_DIR = Path(".")

def find_file(candidate_names, data_dir=DATA_DIR):
    """
    Looks for a file using a short list of possible names.
    This makes the notebook robust to files like school_covariates.csv
    versus school_covariates(2).csv.
    """
    for name in candidate_names:
        path = data_dir / name
        if path.exists():
            return path
    
    raise FileNotFoundError(
        "Could not find any of these files in "
        f"{data_dir.resolve()}:\n" + "\n".join(candidate_names)
    )

PATHS = {
    "school": find_file(["school_covariates.csv", "school_covariates(2).csv"]),
    "district": find_file(["district_covariates.csv", "district_covariates(2).csv"]),
    "train": find_file(["scores_training.csv", "scores_training(2).csv"]),
    "test": find_file(["scores_test.csv", "scores_test(2).csv"]),
}

school_covariates = pd.read_csv(PATHS["school"])
district_covariates = pd.read_csv(PATHS["district"])
scores_training = pd.read_csv(PATHS["train"])
scores_test = pd.read_csv(PATHS["test"])

# Preserve ID and categorical columns as strings.
STRING_COLS = [
    "ASSESSMENT_ID", "SCHOOL", "DISTRICT", "COUNTY",
    "SUBGROUP_NAME", "ASSESSMENT_NAME", "DISTRICT_TYPE", "REGION"
]

for df in [school_covariates, district_covariates, scores_training, scores_test]:
    for col in STRING_COLS:
        if col in df.columns:
            df[col] = df[col].astype("string")

print("Loaded files:")
for key, path in PATHS.items():
    print(f"  {key:8s}: {path}")

def raw_summary(name, df):
    return {
        "table": name,
        "rows": df.shape[0],
        "cols": df.shape[1],
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
        "object_or_string_cols": int(
            df.select_dtypes(include=["object", "string"]).shape[1]
        ),
        "numeric_cols": int(df.select_dtypes(include=[np.number]).shape[1]),
    }

summary = pd.DataFrame([
    raw_summary("school_covariates", school_covariates),
    raw_summary("district_covariates", district_covariates),
    raw_summary("scores_training", scores_training),
    raw_summary("scores_test", scores_test),
])

print("\nRaw table summary:")
print(summary.to_string(index=False))

print("\nKey checks:")
print("school_covariates['SCHOOL'] unique:      ", school_covariates["SCHOOL"].is_unique)
print("district_covariates['DISTRICT'] unique:  ", district_covariates["DISTRICT"].is_unique)
print("scores_training['ASSESSMENT_ID'] unique: ", scores_training["ASSESSMENT_ID"].is_unique)
print("scores_test['ASSESSMENT_ID'] unique:     ", scores_test["ASSESSMENT_ID"].is_unique)

print("\nTarget checks:")
print("'PERCENT_PROFICIENT' in training:", "PERCENT_PROFICIENT" in scores_training.columns)
print("'PERCENT_PROFICIENT' in test:    ", "PERCENT_PROFICIENT" in scores_test.columns)

train_school_coverage = scores_training["SCHOOL"].isin(school_covariates["SCHOOL"]).mean()
test_school_coverage = scores_test["SCHOOL"].isin(school_covariates["SCHOOL"]).mean()

school_districts = school_covariates[["SCHOOL", "DISTRICT"]].drop_duplicates()

train_district_coverage = (
    scores_training[["SCHOOL"]]
    .drop_duplicates()
    .merge(school_districts, on="SCHOOL", how="left")["DISTRICT"]
    .isin(district_covariates["DISTRICT"])
    .mean()
)

test_district_coverage = (
    scores_test[["SCHOOL"]]
    .drop_duplicates()
    .merge(school_districts, on="SCHOOL", how="left")["DISTRICT"]
    .isin(district_covariates["DISTRICT"])
    .mean()
)

print("\nJoin coverage:")
print(f"training rows with SCHOOL in school_covariates: {train_school_coverage:.4f}")
print(f"test rows with SCHOOL in school_covariates:     {test_school_coverage:.4f}")
print(f"training unique schools with DISTRICT data:     {train_district_coverage:.4f}")
print(f"test unique schools with DISTRICT data:         {test_district_coverage:.4f}")

print("\nTarget summary:")
print(scores_training["PERCENT_PROFICIENT"].describe().to_string())

print("\nSoftware:")
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("Random state:", RANDOM_STATE)

Loaded files:
  school  : school_covariates.csv
  district: district_covariates.csv
  train   : scores_training.csv
  test    : scores_test.csv

Raw table summary:
              table   rows  cols  duplicate_rows  missing_cells  object_or_string_cols  numeric_cols
  school_covariates   4754    52               0          26460                      5            47
district_covariates    674     6               0              0                      1             5
    scores_training 144921     6               0              0                      4             2
        scores_test  48307     5               0              0                      4             1

Key checks:
school_covariates['SCHOOL'] unique:       True
district_covariates['DISTRICT'] unique:   True
scores_training['ASSESSMENT_ID'] unique:  True
scores_test['ASSESSMENT_ID'] unique:      True

Target checks:
'PERCENT_PROFICIENT' in training: True
'PERCENT_PROFICIENT' in test:     False

Join coverage:
training rows with 

In [2]:
# ============================================================
# 1. Merge datasets
# ============================================================

# Merge school covariates
train_full = scores_training.merge(
    school_covariates,
    on="SCHOOL",
    how="left",
    validate="many_to_one"
)

test_full = scores_test.merge(
    school_covariates,
    on="SCHOOL",
    how="left",
    validate="many_to_one"
)

# Merge district covariates
train_full = train_full.merge(
    district_covariates,
    on="DISTRICT",
    how="left",
    validate="many_to_one"
)

test_full = test_full.merge(
    district_covariates,
    on="DISTRICT",
    how="left",
    validate="many_to_one"
)

print("train_full shape:", train_full.shape)
print("test_full shape:", test_full.shape)

train_full shape: (144921, 62)
test_full shape: (48307, 61)


In [3]:
# ============================================================
# 2. Missingness overview
# ============================================================

def missing_report(df):
    miss = df.isna().sum()
    miss = miss[miss > 0].sort_values(ascending=False)
    
    report = pd.DataFrame({
        "missing_count": miss,
        "missing_pct": (miss / len(df)) * 100
    })
    
    return report

train_missing = missing_report(train_full)
test_missing = missing_report(test_full)

print("Train missing columns:", train_missing.shape[0])
print("Test missing columns:", test_missing.shape[0])

print("\nTop 15 missing (train):")
print(train_missing.head(15))

print("\nTop 15 missing (test):")
print(test_missing.head(15))

Train missing columns: 52
Test missing columns: 52

Top 15 missing (train):
                                                    missing_count  missing_pct
TEACHER_TURNOVER_RATE                                      134136    92.558014
KINDERGARTEN_AVERAGE_CLASS_SIZE                            103467    71.395450
GRADE_1_AVERAGE_CLASS_SIZE                                 102899    71.003512
GRADE_2_AVERAGE_CLASS_SIZE                                 102779    70.920709
HISTORY_GOVERNMENT_AND_GEOGRAPHY_AVERAGE_CLASS_...          89314    61.629439
PERCENT_DROPOUT                                             16061    11.082590
PERCENT_GED                                                 16061    11.082590
PERCENT_STILL_ENROLLED                                      16061    11.082590
PERCENT_NON_DIPLOMA                                         16061    11.082590
PERCENT_DIPLOMA                                             16061    11.082590
SCIENCE_AVERAGE_CLASS_SIZE                             

In [4]:
# ============================================================
# 3. School-level missingness check
# ============================================================

# Example column: ATTENDANCE_RATE (you can change later)
col = "ATTENDANCE_RATE"

train_missing_schools = train_full[train_full[col].isna()]["SCHOOL"].nunique()
test_missing_schools = test_full[test_full[col].isna()]["SCHOOL"].nunique()

train_missing_rows = train_full[col].isna().sum()
test_missing_rows = test_full[col].isna().sum()

print("Using column:", col)

print("\nTrain rows missing:", train_missing_rows)
print("Train unique schools missing:", train_missing_schools)

print("\nTest rows missing:", test_missing_rows)
print("Test unique schools missing:", test_missing_schools)

Using column: ATTENDANCE_RATE

Train rows missing: 3036
Train unique schools missing: 94

Test rows missing: 983
Test unique schools missing: 93


In [5]:
# ============================================================
# 4. Build X and y + preserve IDs
# ============================================================

TARGET = "PERCENT_PROFICIENT"
ID_COL = "ASSESSMENT_ID"

# Save IDs separately (needed for submission later)
train_ids = train_full[ID_COL].copy()
test_ids = test_full[ID_COL].copy()

# Target
y_train = train_full[TARGET].copy()

# Drop target from features
X_train = train_full.drop(columns=[TARGET])
X_test = test_full.copy()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)

X_train shape: (144921, 61)
X_test shape: (48307, 61)
y_train shape: (144921,)


In [6]:
# ============================================================
# 5. Column typing
# ============================================================

ID_COL = "ASSESSMENT_ID"

# Identify column types
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "string"]).columns.tolist()

# Remove ID from categorical if present
if ID_COL in categorical_cols:
    categorical_cols.remove(ID_COL)

# High-cardinality (simple rule: > 50 unique values)
high_cardinality_cols = [
    col for col in categorical_cols
    if X_train[col].nunique() > 50
]

low_cardinality_cols = [
    col for col in categorical_cols
    if col not in high_cardinality_cols
]

print("Numeric cols:", len(numeric_cols))
print("Categorical cols:", len(categorical_cols))
print("High-cardinality cols:", high_cardinality_cols)
print("Low-cardinality cols:", low_cardinality_cols)

Numeric cols: 53
Categorical cols: 7
High-cardinality cols: ['SCHOOL', 'DISTRICT', 'COUNTY']
Low-cardinality cols: ['SUBGROUP_NAME', 'ASSESSMENT_NAME', 'DISTRICT_TYPE', 'REGION']


In [7]:
# ============================================================
# 6A. Frequency encoding (safe, no leakage)
# ============================================================

X_train_proc = X_train.copy()
X_test_proc = X_test.copy()

freq_encoding_cols = []

for col in high_cardinality_cols:
    freq_map = X_train_proc[col].value_counts(dropna=False)
    
    new_col = col + "_freq"
    X_train_proc[new_col] = X_train_proc[col].map(freq_map).astype(float)
    X_test_proc[new_col] = X_test_proc[col].map(freq_map).fillna(0).astype(float)
    
    freq_encoding_cols.append(new_col)

print("Added frequency columns:", freq_encoding_cols)
print("X_train_proc shape:", X_train_proc.shape)
print("X_test_proc shape:", X_test_proc.shape)

Added frequency columns: ['SCHOOL_freq', 'DISTRICT_freq', 'COUNTY_freq']
X_train_proc shape: (144921, 64)
X_test_proc shape: (48307, 64)


In [8]:
# ============================================================
# 6B. Missing indicators + median imputation (more robust than mean)
# ============================================================

numeric_cols_extended = numeric_cols + freq_encoding_cols

missing_indicator_cols = []

for col in numeric_cols_extended:
    if X_train_proc[col].isna().sum() > 0:
        new_col = col + "_missing"
        
        X_train_proc[new_col] = X_train_proc[col].isna().astype(int)
        X_test_proc[new_col] = X_test_proc[col].isna().astype(int)
        
        median_val = X_train_proc[col].median()
        
        X_train_proc[col] = X_train_proc[col].fillna(median_val)
        X_test_proc[col] = X_test_proc[col].fillna(median_val)
        
        missing_indicator_cols.append(new_col)

print("Missing indicators added:", len(missing_indicator_cols))
print("New shape:", X_train_proc.shape)

Missing indicators added: 52
New shape: (144921, 116)


/var/folders/cb/fffq6hxx2qvgkbh5yps70l_w0000gn/T/ipykernel_48297/1325974498.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train_proc[new_col] = X_train_proc[col].isna().astype(int)
/var/folders/cb/fffq6hxx2qvgkbh5yps70l_w0000gn/T/ipykernel_48297/1325974498.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test_proc[new_col] = X_test_proc[col].isna().astype(int)
/var/folders/cb/fffq6hxx2qvgkbh5yps70l_w0000gn/T/ipykernel_48297/1325974498.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the 

In [9]:
# ============================================================
# 6C. Drop raw high-cardinality categorical columns
# ============================================================

X_train_proc = X_train_proc.drop(columns=high_cardinality_cols)
X_test_proc = X_test_proc.drop(columns=high_cardinality_cols)

print("Shape after dropping high-cardinality cols:", X_train_proc.shape)

Shape after dropping high-cardinality cols: (144921, 113)


### Feature Encoding: High-Cardinality Variables

The raw high-cardinality categorical variables `SCHOOL`, `DISTRICT`, and `COUNTY` were removed from the modeling matrix after frequency encodings were created for them.

This prevents the model from directly using raw ID-like categorical labels while still preserving useful information about how frequently each school, district, or county appears in the training data.

After dropping these raw columns, the feature matrix has 115 columns.

In [10]:
# ============================================================
# 6D. One-hot encode low-cardinality categorical variables
# ============================================================

X_train_proc = pd.get_dummies(
    X_train_proc,
    columns=low_cardinality_cols,
    drop_first=False
)

X_test_proc = pd.get_dummies(
    X_test_proc,
    columns=low_cardinality_cols,
    drop_first=False
)

# Align columns (important!)
X_train_proc, X_test_proc = X_train_proc.align(X_test_proc, join="left", axis=1, fill_value=0)

print("Final X_train shape:", X_train_proc.shape)
print("Final X_test shape:", X_test_proc.shape)

Final X_train shape: (144921, 163)
Final X_test shape: (48307, 163)


### Final Feature Matrix

After full preprocessing:

- High-cardinality variables (`SCHOOL`, `DISTRICT`, `COUNTY`) were replaced with frequency encodings
- Missing values were handled via:
  - median imputation
  - explicit missingness indicator variables
- Low-cardinality categorical variables were one-hot encoded:
  - `SUBGROUP_NAME`, `ASSESSMENT_NAME`, `DISTRICT_TYPE`, `REGION`

Final dimensions:
- Training set: 144,921 rows × 165 features
- Test set: 48,307 rows × 165 features

The feature space is now fully numeric, aligned between train and test, and ready for modeling.

In [11]:
# ============================================================
# Create modeling copy WITHOUT ID (non-destructive)
# ============================================================

X_train_proc_model = X_train_proc.drop(columns=["ASSESSMENT_ID"])
X_test_proc_model = X_test_proc.drop(columns=["ASSESSMENT_ID"])

print("Modeling shape:", X_train_proc_model.shape)

Modeling shape: (144921, 162)


### Modeling Dataset

A separate modeling dataset was created by removing the identifier column `ASSESSMENT_ID`.

Final modeling dimensions:
- Training set: 144,921 rows × 164 features

This ensures that all features used for modeling are numeric or boolean, and that no identifier-based leakage occurs.

In [12]:
# ============================================================
# 7A. Simple Linear Regression (one feature at a time)
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_proc_model, y_train, test_size=0.2, random_state=9890
)

simple_lr_results = []

for col in X_train_proc_model.columns:
    model = LinearRegression()
    model.fit(X_tr[[col]], y_tr)
    
    train_pred = model.predict(X_tr[[col]])
    val_pred = model.predict(X_val[[col]])
    
    simple_lr_results.append({
        "feature": col,
        "train_mse": mean_squared_error(y_tr, train_pred),
        "val_mse": mean_squared_error(y_val, val_pred)
    })

simple_lr_results = pd.DataFrame(simple_lr_results).sort_values("val_mse")

print(simple_lr_results.head(30).to_string(index=False))

                                                    feature  train_mse    val_mse
                         PERCENT_ECONOMICALLY_DISADVANTAGED 570.735244 581.066288
                                         PERCENT_FREE_LUNCH 575.082361 584.868424
                                            PERCENT_DIPLOMA 621.742550 626.029451
                                     PERCENT_STILL_ENROLLED 637.471784 641.000299
                                           PERCENT_HOMELESS 640.796399 644.070716
                                              PERCENT_BLACK 648.963234 652.648495
                                  PERCENT_WITH_DISABILITIES 650.316873 653.054139
                           PERCENT_ENGLISH_LANGUAGE_LEANERS 648.460937 655.348885
                                            ATTENDANCE_RATE 650.513825 657.793054
                                              PERCENT_WHITE 653.127789 658.601548
                                            PERCENT_DROPOUT 653.769449 661.927501
                

In [13]:
# ============================================================
# Extract best simple linear regression feature properly
# ============================================================

best_feature_row = simple_lr_results.loc[simple_lr_results["val_mse"].idxmin()]

print("Best single-feature model:")
print(best_feature_row)

Best single-feature model:
feature      PERCENT_ECONOMICALLY_DISADVANTAGED
train_mse                            570.735244
val_mse                              581.066288
Name: 43, dtype: object


### Simple Linear Regression Baseline

Each feature was tested individually in a simple linear regression model. This creates a baseline ranking of single predictors before fitting larger multiple regression models.

The goal is not to select the final model from one feature, but to identify which variables have the strongest individual linear relationship with `PERCENT_PROFICIENT`.

### Best Simple Linear Regression Feature

The best single-feature model was:

- Feature: `PERCENT_ECONOMICALLY_DISADVANTAGED`
- Train MSE: 570.74  
- Validation MSE: 581.07  

Interpretation:
- Socioeconomic disadvantage is the strongest standalone predictor of `PERCENT_PROFICIENT`.
- However, the error (~581) is substantially higher than the multiple linear regression model (~312), indicating that no single variable explains the outcome well.
- This confirms that predictive power in the dataset is distributed across multiple correlated features rather than dominated by a single factor.

Conclusion:
Simple linear regression provides insight into marginal relationships but is insufficient for accurate prediction on its own.

### Simple Linear Regression Insights

The strongest individual predictors of `PERCENT_PROFICIENT` are:

- Socioeconomic indicators:
  - `PERCENT_ECONOMICALLY_DISADVANTAGED`
  - `PERCENT_FREE_LUNCH`
- Academic outcomes:
  - `PERCENT_DIPLOMA`
  - `PERCENT_STILL_ENROLLED`
- Demographics:
  - `PERCENT_BLACK`, `PERCENT_WHITE`, `PERCENT_HISPANIC`
- Vulnerability indicators:
  - `PERCENT_HOMELESS`, `PERCENT_WITH_DISABILITIES`
- Attendance:
  - `ATTENDANCE_RATE`

Key observations:
- Socioeconomic disadvantage is the strongest single predictor.
- Many top features are highly correlated (e.g., free lunch vs economic disadvantage).
- Missingness indicators appear, confirming that missing data carries signal.
- Frequency-encoded variables are not dominant individually, suggesting entity effects are weaker in isolation.

Conclusion:
Simple linear regression highlights strong marginal relationships but does not account for interactions or multicollinearity.

In [14]:
# ============================================================
# 7B. Multiple Linear Regression (clean baseline)
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Fresh split (consistent with 7A)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_proc_model, y_train, test_size=0.2, random_state=9890
)

model = LinearRegression()
model.fit(X_tr, y_tr)

train_pred = model.predict(X_tr)
val_pred = model.predict(X_val)

train_mse = mean_squared_error(y_tr, train_pred)
val_mse = mean_squared_error(y_val, val_pred)

print("Train MSE:", train_mse)
print("Validation MSE:", val_mse)

Train MSE: 305.12409327017724
Validation MSE: 312.6016702920729


### Multiple Linear Regression Baseline

A multiple linear regression model was fitted using all available features.

Results:
- Train MSE: 305.12  
- Validation MSE: 312.60  

Interpretation:
- The gap between training and validation error is small, indicating minimal overfitting.
- The model performs substantially better than the best simple linear regression (~581 MSE), showing that predictive power is distributed across multiple features.
- The relatively low validation error suggests that linear relationships capture a large portion of the underlying structure in the data.

Conclusion:
Multiple linear regression provides a strong and stable baseline for evaluating more complex models.

In [15]:
# Select top 10 features from simple LR
top_features = simple_lr_results.nsmallest(10, "val_mse")["feature"].tolist()

print(top_features)

['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE']


In [16]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=False)

X_tr_poly = poly.fit_transform(X_tr[top_features])
X_val_poly = poly.transform(X_val[top_features])

print("Poly feature shape:", X_tr_poly.shape)

Poly feature shape: (115936, 65)


In [17]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

model = LinearRegression()
model.fit(X_tr_poly, y_tr)

train_pred = model.predict(X_tr_poly)
val_pred = model.predict(X_val_poly)

print("Polynomial Regression:")
print("Train MSE:", mean_squared_error(y_tr, train_pred))
print("Validation MSE:", mean_squared_error(y_val, val_pred))

Polynomial Regression:
Train MSE: 507.61481933233904
Validation MSE: 515.1911286593601


### Polynomial Regression

Polynomial regression (degree 2) was applied to the top 10 features identified from simple linear regression.

Results:
- Train MSE: 507.61  
- Validation MSE: 515.19  

Interpretation:
- Performance is significantly worse than multiple linear regression (~312 MSE).
- This indicates that restricting the model to a small subset of features removes important predictive information.
- The polynomial expansion does not compensate for the loss of breadth in the feature space.

Conclusion:
The dataset appears to benefit more from combining many features linearly rather than modeling nonlinear relationships among a small subset of variables. Polynomial regression is not effective in this setting.

In [18]:
top5_features = simple_lr_results.nsmallest(5, "val_mse")["feature"].tolist()
print(top5_features)

['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS']


In [19]:
from itertools import combinations

interaction_cols = []

X_tr_int = X_tr.copy()
X_val_int = X_val.copy()

for f1, f2 in combinations(top5_features, 2):
    new_col = f"{f1}_x_{f2}"
    
    X_tr_int[new_col] = X_tr[f1] * X_tr[f2]
    X_val_int[new_col] = X_val[f1] * X_val[f2]
    
    interaction_cols.append(new_col)

print("Number of interaction features added:", len(interaction_cols))

Number of interaction features added: 10


In [20]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

model = LinearRegression()
model.fit(X_tr_int, y_tr)

train_pred = model.predict(X_tr_int)
val_pred = model.predict(X_val_int)

print("Interaction Model:")
print("Train MSE:", mean_squared_error(y_tr, train_pred))
print("Validation MSE:", mean_squared_error(y_val, val_pred))

Interaction Model:
Train MSE: 304.3250676749579
Validation MSE: 311.955526746157


### Interaction Model

Pairwise interaction terms were added between the top 5 features identified from simple linear regression.

Results:
- Train MSE: 304.33  
- Validation MSE: 311.96  

Interpretation:
- The interaction model slightly improves performance compared to multiple linear regression (~312.60 → ~311.96).
- The improvement is marginal, suggesting that most of the predictive structure is already captured by additive linear effects.
- Interactions contribute some additional signal but are not a dominant factor in this dataset.

Conclusion:
While interaction terms provide a small improvement, the dataset is largely driven by additive relationships rather than strong nonlinear interactions.

In [21]:
# ============================================================
# 8A. Build controlled candidate feature spaces
# ============================================================

from itertools import combinations

# Use strongest marginal predictors as candidates for engineered terms
top10_features = simple_lr_results.nsmallest(10, "val_mse")["feature"].tolist()
top5_features = simple_lr_results.nsmallest(5, "val_mse")["feature"].tolist()

# Base space
X_tr_base = X_tr.copy()
X_val_base = X_val.copy()

# Base + polynomial squares for top10
X_tr_polyspace = X_tr.copy()
X_val_polyspace = X_val.copy()

poly_cols = []

for col in top10_features:
    new_col = col + "_squared"
    X_tr_polyspace[new_col] = X_tr[col] ** 2
    X_val_polyspace[new_col] = X_val[col] ** 2
    poly_cols.append(new_col)

# Base + pairwise interactions for top5
X_tr_intspace = X_tr.copy()
X_val_intspace = X_val.copy()

interaction_cols = []

for f1, f2 in combinations(top5_features, 2):
    new_col = f"{f1}_x_{f2}"
    X_tr_intspace[new_col] = X_tr[f1] * X_tr[f2]
    X_val_intspace[new_col] = X_val[f1] * X_val[f2]
    interaction_cols.append(new_col)

# Base + polynomial + interactions
X_tr_combined = X_tr_polyspace.copy()
X_val_combined = X_val_polyspace.copy()

for col in interaction_cols:
    X_tr_combined[col] = X_tr_intspace[col]
    X_val_combined[col] = X_val_intspace[col]

feature_spaces = {
    "base": (X_tr_base, X_val_base),
    "base_plus_poly": (X_tr_polyspace, X_val_polyspace),
    "base_plus_interactions": (X_tr_intspace, X_val_intspace),
    "base_plus_poly_interactions": (X_tr_combined, X_val_combined)
}

print("Top 10 features used for squares:")
print(top10_features)

print("\nTop 5 features used for interactions:")
print(top5_features)

print("\nPolynomial columns added:", len(poly_cols))
print("Interaction columns added:", len(interaction_cols))

print("\nFeature space shapes:")
for name, (X_train_space, X_val_space) in feature_spaces.items():
    print(name, X_train_space.shape, X_val_space.shape)

Top 10 features used for squares:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE']

Top 5 features used for interactions:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS']

Polynomial columns added: 10
Interaction columns added: 10

Feature space shapes:
base (115936, 162) (28985, 162)
base_plus_poly (115936, 172) (28985, 172)
base_plus_interactions (115936, 172) (28985, 172)
base_plus_poly_interactions (115936, 182) (28985, 182)


In [ ]:
# ============================================================
# 8B. Forward stepwise selection across feature spaces
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import pandas as pd

def forward_stepwise(X_train_space, X_val_space, y_train, y_val, max_features=30):
    remaining = list(X_train_space.columns)
    selected = []
    rows = []
    best_mse = float("inf")
    
    while remaining and len(selected) < max_features:
        candidates = []
        
        for feature in remaining:
            trial_features = selected + [feature]
            
            model = LinearRegression()
            model.fit(X_train_space[trial_features], y_train)
            
            val_pred = model.predict(X_val_space[trial_features])
            val_mse = mean_squared_error(y_val, val_pred)
            
            candidates.append((feature, val_mse))
        
        best_feature, candidate_mse = min(candidates, key=lambda x: x[1])
        
        if candidate_mse < best_mse:
            selected.append(best_feature)
            remaining.remove(best_feature)
            best_mse = candidate_mse
            
            rows.append({
                "num_features": len(selected),
                "feature_added": best_feature,
                "val_mse": best_mse
            })
        else:
            break
    
    return pd.DataFrame(rows), selected

forward_summary = {}

for space_name, (X_train_space, X_val_space) in feature_spaces.items():
    results, selected = forward_stepwise(
        X_train_space, X_val_space, y_tr, y_val, max_features=30
    )
    
    forward_summary[space_name] = {
        "results": results,
        "selected_features": selected,
        "best_val_mse": results["val_mse"].min() if len(results) > 0 else None,
        "best_num_features": results.loc[results["val_mse"].idxmin(), "num_features"] if len(results) > 0 else None
    }
    
    print("\n" + "=" * 80)
    print(space_name)
    print("=" * 80)
    print(results.tail(10).to_string(index=False))
    print("Best validation MSE:", forward_summary[space_name]["best_val_mse"])


base
 num_features                                       feature_added    val_mse
           21                                   DISTRICT_TYPE_NYC 356.639984
           22                     ASSESSMENT_NAME_RegentsScience8 352.619203
           23                               ASSESSMENT_NAME_MATH4 348.915113
           24                               PERCENT_REDUCED_LUNCH 346.330326
           25           ASSESSMENT_NAME_Regents Phy Set/Chemistry 344.123928
           26        ASSESSMENT_NAME_Regents Common Core Geometry 341.579203
           27             ASSESSMENT_NAME_Regents Phy Set/Physics 338.762894
           28 HISTORY_GOVERNMENT_AND_GEOGRAPHY_AVERAGE_CLASS_SIZE 336.466988
           29                               ASSESSMENT_NAME_MATH7 334.174799
           30                               ASSESSMENT_NAME_MATH3 332.314849
Best validation MSE: 332.3148493948422

base_plus_poly
 num_features                                       feature_added    val_mse
           21  

### Forward Stepwise Design Choice

Forward stepwise selection was used to evaluate whether a smaller subset of predictors can approach the performance of the full multiple linear regression model.

Because the full feature space contains 164–184 predictors depending on the feature set, unrestricted stepwise selection would be computationally expensive and would gradually reconstruct the full model. To keep the procedure tractable and focused on model parsimony, the search was capped at 30 selected features.

This cap is not intended to imply that 30 is theoretically optimal. Instead, it provides a practical stopping limit that allows us to examine whether most predictive gains occur early in the selection path. If validation MSE is still improving near 30 features, the cap can be increased later.

### Forward Stepwise Selection

Forward stepwise selection was applied across multiple feature spaces, including:
- Base feature space
- Base + polynomial terms
- Base + interaction terms
- Base + polynomial + interaction terms

Results:
- Best validation MSE (base): 332.31  
- Best validation MSE (poly): 329.66  

Interpretation:
- All stepwise models perform significantly worse than the full multiple linear regression model (~312.60).
- This indicates that predictive performance relies on combining a large number of features rather than selecting a small subset.
- Polynomial and interaction features provide only marginal improvements within the stepwise framework.

Conclusion:
Subset selection via forward stepwise is not effective for this dataset. The data exhibits a high-dimensional additive structure where many weak predictors contribute jointly. Methods that retain all features while controlling complexity (e.g., Ridge or Lasso) are more appropriate.

In [27]:
# ============================================================
# Backward Stepwise (controlled)
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import pandas as pd

def backward_stepwise(X_train_space, X_val_space, y_train, y_val, max_removals=30):
    
    selected = list(X_train_space.columns)
    results = []
    
    # Initial model (full)
    model = LinearRegression()
    model.fit(X_train_space[selected], y_train)
    val_pred = model.predict(X_val_space[selected])
    best_mse = mean_squared_error(y_val, val_pred)
    
    results.append({
        "num_features": len(selected),
        "removed_feature": None,
        "val_mse": best_mse
    })
    
    for _ in range(max_removals):
        candidates = []
        
        for feature in selected:
            trial_features = [f for f in selected if f != feature]
            
            model = LinearRegression()
            model.fit(X_train_space[trial_features], y_train)
            
            val_pred = model.predict(X_val_space[trial_features])
            mse = mean_squared_error(y_val, val_pred)
            
            candidates.append((feature, mse))
        
        worst_feature, candidate_mse = min(candidates, key=lambda x: x[1])
        
        if candidate_mse <= best_mse:
            selected.remove(worst_feature)
            best_mse = candidate_mse
            
            results.append({
                "num_features": len(selected),
                "removed_feature": worst_feature,
                "val_mse": best_mse
            })
        else:
            break
    
    return pd.DataFrame(results)


backward_summary = {}

for name, (X_train_space, X_val_space) in feature_spaces.items():
    print("\n" + "="*80)
    print(name)
    print("="*80)
    
    results = backward_stepwise(X_train_space, X_val_space, y_tr, y_val, max_removals=30)
    
    backward_summary[name] = results
    
    print(results.tail(10).to_string(index=False))
    print("Best val MSE:", results["val_mse"].min())


base
 num_features                       removed_feature    val_mse
          153                         PERCENT_ASIAN 312.378267
          152          ASSESSMENT_NAME_RegentsMath8 312.378267
          151          ASSESSMENT_NAME_RegentsMath7 312.367797
          150                  REGION_Southern Tier 312.367797
          149                    REGION_Long Island 312.357865
          148                  REGION_Mohawk Valley 312.353135
          147 PERCENT_OF_STUDENTS_SUSPENDED_missing 312.353135
          146               ATTENDANCE_RATE_missing 312.353135
          145                  SUBGROUP_NAME_Female 312.353135
          144                      N_PUPILS_missing 312.353135
Best val MSE: 312.3531354590935

base_plus_poly
 num_features                               removed_feature    val_mse
          167                                   PERCENT_GED 308.801652
          166                                      GRADE_07 308.801547
          165 ASSESSMENT_NAME_Regents Co

In [27]:
# ============================================================
# Hybrid Stepwise (forward + backward cleanup)
# ============================================================

def hybrid_stepwise(X_train_space, X_val_space, y_train, y_val, max_features=30):
    
    remaining = list(X_train_space.columns)
    selected = []
    results = []
    
    best_mse = float("inf")
    
    while remaining and len(selected) < max_features:
        
        # Forward step
        candidates = []
        
        for feature in remaining:
            trial_features = selected + [feature]
            
            model = LinearRegression()
            model.fit(X_train_space[trial_features], y_train)
            
            val_pred = model.predict(X_val_space[trial_features])
            mse = mean_squared_error(y_val, val_pred)
            
            candidates.append((feature, mse))
        
        best_feature, candidate_mse = min(candidates, key=lambda x: x[1])
        
        if candidate_mse < best_mse:
            selected.append(best_feature)
            remaining.remove(best_feature)
            best_mse = candidate_mse
        else:
            break
        
        # Backward cleanup step
        improved = True
        while improved and len(selected) > 1:
            improved = False
            
            for feature in selected:
                trial_features = [f for f in selected if f != feature]
                
                model = LinearRegression()
                model.fit(X_train_space[trial_features], y_train)
                
                val_pred = model.predict(X_val_space[trial_features])
                mse = mean_squared_error(y_val, val_pred)
                
                if mse < best_mse:
                    selected.remove(feature)
                    best_mse = mse
                    improved = True
                    break
        
        results.append({
            "num_features": len(selected),
            "val_mse": best_mse
        })
    
    return pd.DataFrame(results)


hybrid_summary = {}

for name, (X_train_space, X_val_space) in feature_spaces.items():
    print("\n" + "="*80)
    print(name)
    print("="*80)
    
    results = hybrid_stepwise(X_train_space, X_val_space, y_tr, y_val, max_features=30)
    
    hybrid_summary[name] = results
    
    print(results.tail(10).to_string(index=False))
    print("Best val MSE:", results["val_mse"].min())


base
 num_features    val_mse
           21 356.639984
           22 352.619203
           23 348.915113
           24 346.330326
           25 344.123928
           26 341.579203
           27 338.762894
           28 336.466988
           29 334.174799
           30 332.314849
Best val MSE: 332.3148493948422

base_plus_poly
 num_features    val_mse
           21 352.226262
           22 348.874109
           23 346.326338
           24 343.776219
           25 341.160388
           26 338.657725
           27 336.137043
           28 333.717583
           29 331.600432
           30 329.660042
Best val MSE: 329.6600419738662

base_plus_interactions
 num_features    val_mse
           21 356.639984
           22 352.619203
           23 348.915113
           24 346.330326
           25 344.123928
           26 341.579203
           27 338.762894
           28 336.466988
           29 334.174799
           30 332.314849
Best val MSE: 332.3148493948422

base_plus_poly_interactions
 num

### Backward and Hybrid Stepwise Selection

Backward and hybrid stepwise selection were evaluated across four controlled feature spaces:

1. Base feature space
2. Base + polynomial terms
3. Base + interaction terms
4. Base + polynomial + interaction terms

Backward stepwise performed better than forward stepwise because it began with the full model and removed only features whose exclusion improved or preserved validation performance.

Best backward stepwise validation MSE values:

- Base: 312.35
- Base + polynomial: 308.77
- Base + interactions: 311.70
- Base + polynomial + interactions: 308.10

The best backward stepwise model was the base + polynomial + interaction model, with validation MSE of 308.10. This improves on the full multiple linear regression baseline of approximately 312.60.

The hybrid stepwise results matched the forward stepwise results exactly, suggesting that the backward cleanup phase did not remove any features after forward additions. Therefore, in this implementation, hybrid stepwise effectively behaved like forward stepwise.

Interpretation:
- Forward stepwise performed worse because it was limited to 30 selected features and could not capture the distributed signal across many predictors.
- Backward stepwise performed better because it retained most of the full feature space while pruning redundant or harmful variables.
- Polynomial terms provided meaningful improvement when added to the full feature space and pruned through backward selection.
- Interaction terms alone provided only modest improvement.

Conclusion:
The strongest linear-model-family result so far is backward stepwise on the combined polynomial + interaction feature space. This suggests that the dataset is mostly additive and high-dimensional, but selected nonlinear terms can improve performance when incorporated carefully.

In [22]:
# ============================================================
# Validation protocol note
# ============================================================

VALIDATION_PROTOCOL = {
    "current_stage": "development_holdout",
    "split_method": "train_test_split",
    "test_size": 0.20,
    "random_state": 9890,
    "final_stage": "cross_validation_on_shortlist",
    "note": (
        "Current MSE values are development validation scores. "
        "Cross-validation will be run later on shortlisted models."
    )
}

VALIDATION_PROTOCOL

{'current_stage': 'development_holdout',
 'split_method': 'train_test_split',
 'test_size': 0.2,
 'random_state': 9890,
 'final_stage': 'cross_validation_on_shortlist',
 'note': 'Current MSE values are development validation scores. Cross-validation will be run later on shortlisted models.'}

### Cross-Validation Strategy

At this stage, models are being evaluated using a fixed train/validation split. These results are useful for rapid model screening, but they should not be treated as final estimates of generalization performance.

Cross-validation will be applied later after the main model families have been tested. This avoids excessive computation during the exploratory phase while still allowing rigorous comparison among serious finalist models.

Current terminology:
- `val_mse`: development holdout validation MSE
- `cv_mse`: cross-validated MSE, to be computed later for shortlisted models

Planned approach:
1. Use the current validation split to screen many model classes quickly.
2. Record all results in a model ledger.
3. Shortlist the strongest models.
4. Run cross-validation on the shortlist.
5. Tune/refine the strongest cross-validated candidates.
6. Select the final model for submission.

For models involving target encoding or feature selection, special care will be needed to avoid leakage. Target encoding should be performed out-of-fold, and feature selection may need to be repeated inside folds if the selected model becomes a serious finalist.

In [24]:
# ============================================================
# 9A. Ridge Regression: two-stage alpha search + scaler comparison
# ============================================================

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Design choices
# ------------------------------------------------------------
# We compare StandardScaler and RobustScaler instead of assuming one.
# We use a two-stage alpha search:
#   1. Broad search across many orders of magnitude
#   2. Fine search around the best alpha from the broad search
#
# Cross-validation is intentionally postponed until later finalist screening.
# These are still development holdout validation results.

scaler_factories = {
    "standard": StandardScaler,
    "robust": RobustScaler
}

broad_alphas = np.logspace(-4, 8, 49)  # 0.0001 to 100,000,000

def evaluate_ridge_grid(X_train_space, X_val_space, y_train, y_val, alphas, scaler_name, scaler_factory, stage):
    rows = []
    
    for alpha in alphas:
        model = make_pipeline(
            scaler_factory(),
            Ridge(alpha=alpha)
        )
        
        model.fit(X_train_space, y_train)
        
        train_pred = model.predict(X_train_space)
        val_pred = model.predict(X_val_space)
        
        rows.append({
            "model_class": "Ridge",
            "stage": stage,
            "scaler": scaler_name,
            "alpha": alpha,
            "train_mse": mean_squared_error(y_train, train_pred),
            "val_mse": mean_squared_error(y_val, val_pred),
            "aic": np.nan,
            "bic": np.nan
        })
    
    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Stage 1: broad search
# ------------------------------------------------------------

ridge_broad_rows = []

for feature_space_name, (X_train_space, X_val_space) in feature_spaces.items():
    for scaler_name, scaler_factory in scaler_factories.items():
        
        result = evaluate_ridge_grid(
            X_train_space=X_train_space,
            X_val_space=X_val_space,
            y_train=y_tr,
            y_val=y_val,
            alphas=broad_alphas,
            scaler_name=scaler_name,
            scaler_factory=scaler_factory,
            stage="broad"
        )
        
        result["feature_space"] = feature_space_name
        ridge_broad_rows.append(result)

ridge_broad_results = pd.concat(ridge_broad_rows, ignore_index=True)

best_broad_by_space_scaler = (
    ridge_broad_results
    .loc[ridge_broad_results.groupby(["feature_space", "scaler"])["val_mse"].idxmin()]
    .sort_values("val_mse")
)

print("Best broad Ridge result by feature space and scaler:")
print(best_broad_by_space_scaler.to_string(index=False))


# ------------------------------------------------------------
# Stage 2: fine search around each broad-search winner
# ------------------------------------------------------------

ridge_fine_rows = []

for _, row in best_broad_by_space_scaler.iterrows():
    feature_space_name = row["feature_space"]
    scaler_name = row["scaler"]
    scaler_factory = scaler_factories[scaler_name]
    best_alpha = row["alpha"]
    
    # Search within +/- 0.75 log10 units around the broad winner.
    # This is about a 5.6x range on either side.
    log_alpha = np.log10(best_alpha)
    fine_low = max(np.log10(broad_alphas.min()), log_alpha - 0.75)
    fine_high = min(np.log10(broad_alphas.max()), log_alpha + 0.75)
    
    fine_alphas = np.logspace(fine_low, fine_high, 31)
    
    X_train_space, X_val_space = feature_spaces[feature_space_name]
    
    result = evaluate_ridge_grid(
        X_train_space=X_train_space,
        X_val_space=X_val_space,
        y_train=y_tr,
        y_val=y_val,
        alphas=fine_alphas,
        scaler_name=scaler_name,
        scaler_factory=scaler_factory,
        stage="fine"
    )
    
    result["feature_space"] = feature_space_name
    ridge_fine_rows.append(result)

ridge_fine_results = pd.concat(ridge_fine_rows, ignore_index=True)


# ------------------------------------------------------------
# Combine broad + fine results and select winners
# ------------------------------------------------------------

ridge_results = pd.concat(
    [ridge_broad_results, ridge_fine_results],
    ignore_index=True
)

best_ridge_by_space = (
    ridge_results
    .loc[ridge_results.groupby("feature_space")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

best_ridge_overall = ridge_results.loc[ridge_results["val_mse"].idxmin()]

print("\nBest Ridge result by feature space:")
print(best_ridge_by_space.to_string(index=False))

print("\nOverall best Ridge result:")
print(best_ridge_overall)

# ------------------------------------------------------------
# Optional warning: if best alpha is at broad grid boundary
# ------------------------------------------------------------

if best_ridge_overall["alpha"] == broad_alphas.min():
    print("\nWARNING: Best alpha is at the minimum broad-grid boundary. Consider expanding lower.")
elif best_ridge_overall["alpha"] == broad_alphas.max():
    print("\nWARNING: Best alpha is at the maximum broad-grid boundary. Consider expanding higher.")

Best broad Ridge result by feature space and scaler:
model_class stage   scaler    alpha  train_mse    val_mse  aic  bic               feature_space
      Ridge broad standard 1.778279 300.906794 308.357271  NaN  NaN base_plus_poly_interactions
      Ridge broad   robust 0.316228 300.891402 308.358944  NaN  NaN base_plus_poly_interactions
      Ridge broad standard 0.000100 301.622160 308.925244  NaN  NaN              base_plus_poly
      Ridge broad   robust 0.000100 301.622160 308.925245  NaN  NaN              base_plus_poly
      Ridge broad standard 0.000100 304.325068 311.955527  NaN  NaN      base_plus_interactions
      Ridge broad   robust 0.000100 304.325068 311.955527  NaN  NaN      base_plus_interactions
      Ridge broad standard 1.778279 305.124297 312.601534  NaN  NaN                        base
      Ridge broad   robust 0.000100 305.124093 312.601671  NaN  NaN                        base

Best Ridge result by feature space:
model_class stage   scaler    alpha  train_mse

### Ridge Regression Results

Ridge regression was evaluated across four controlled feature spaces:

1. Base feature space
2. Base + polynomial terms
3. Base + interaction terms
4. Base + polynomial + interaction terms

Both `StandardScaler` and `RobustScaler` were compared, and alpha was tuned using a two-stage holdout-validation search.

Best Ridge model:
- Feature space: base + polynomial + interactions
- Scaler: StandardScaler
- Alpha: 1.41
- Train MSE: 300.90
- Validation MSE: 308.36

Interpretation:
- Ridge performs best on the combined polynomial + interaction feature space.
- StandardScaler slightly outperforms RobustScaler.
- Ridge improves over the base multiple linear regression model but does not quite outperform the best backward stepwise model.
- The improvement appears to come mainly from the engineered feature space rather than from shrinkage alone.

Conclusion:
Ridge is a strong regularized linear model, but the current best linear-family model remains backward stepwise on the combined polynomial + interaction feature space, with validation MSE around 308.10.

In [25]:
# ============================================================
# 10A. Lasso Regression: data-driven two-stage alpha search
# ============================================================

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import time
import warnings
from sklearn.exceptions import ConvergenceWarning

# ------------------------------------------------------------
# Design choices
# ------------------------------------------------------------
# - Feature spaces: all four controlled spaces
# - Scalers: StandardScaler and RobustScaler
# - Alpha search:
#     Stage 1: broad data-driven search from alpha_max downward
#     Stage 2: fine search around the best broad alpha
# - Metric: development holdout validation MSE
# - Extra output: number of nonzero coefficients
# - CV is intentionally postponed until finalist screening

scaler_factories = {  #This means every feature space will be tested twice: once with standard scaling and once with robust scaling.
    "standard": StandardScaler,
    "robust": RobustScaler
}
# The constants below control the search over Lasso’s tuning parameter,
LASSO_BROAD_GRID_SIZE = 25 # means the first search tries 25 alpha values per feature-space/scaler combination.
LASSO_FINE_GRID_SIZE = 21 # means the second search tries 21 more alpha values near the best one from the broad search.
LASSO_MIN_ALPHA_RATIO = 1e-4 # means the broad search goes from alpha_max down to alpha_max * 0.0001
LASSO_FINE_WIDTH_LOG10 = 0.5 # means the fine search checks values about 3.16 times above and below the best broad alpha, because 10^0.5 ≈ 3.16.

LASSO_MAX_ITER = 30000 # gives the solver many iterations to converge. Lasso can be harder to optimize than Ridge, especially with correlated predictors.
LASSO_TOL = 1e-4 # controls how precise the optimization has to be before it stops.

def compute_lasso_alpha_max(X_scaled, y): # This computes the largest alpha value worth trying.
    """
    Computes alpha_max for sklearn's Lasso objective:
        (1 / (2n)) * ||y - Xw||^2 + alpha * ||w||_1

    At alpha >= alpha_max, all coefficients are zero.
    """
    y_arr = np.asarray(y, dtype=float)
    y_centered = y_arr - y_arr.mean()
    n = X_scaled.shape[0] # the number of training samples (rows) in your dataset.
    alpha_max = np.max(np.abs(X_scaled.T @ y_centered)) / n  # X_scaled.T @ y_centered (@ means matrix muliplication) measures how strongly each feature is associated with the centered target. The feature with the strongest relationship to the target determines the largest penalty needed to force every coefficient to zero. This is smarter than choosing a random alpha grid. Lasso’s useful alpha range depends heavily on the scale of the features and the target. So instead of searching arbitrary values like 0.001 to 1000, this code builds a custom alpha range for each feature space and scaler. "How strongly does this feature move with the target?"
    # If a feature increases when y increases → big positive value
    # If a feature decreases when y increases → big negative value
    # If a feature is unrelated → value near 0
    return float(alpha_max)
    #the feature most strongly related to the target, That’s exactly what determines the largest alpha where Lasso wipes everything out.

def fit_lasso_path( # This function fits many Lasso models over a list of alpha values and records the results.
    X_train_space,
    X_val_space,
    y_train,
    y_val,
    alphas,
    scaler_name,
    scaler_factory,
    feature_space_name,
    stage
):
    """
    Fits Lasso models over a path of alphas using warm starts.
    Alphas should be sorted from largest to smallest.
    """
    rows = []
    # the scaler below is fit only on the training data, we learn the scaling parameters from the training split, then later apply the same transformation to the validation split.
    scaler = scaler_factory()
    X_train_scaled = scaler.fit_transform(X_train_space)
    X_val_scaled = scaler.transform(X_val_space)
    
    model = Lasso(
        alpha=alphas[0],
        fit_intercept=True, # means the model includes an intercept term. The intercept is not penalized by Lasso.
        max_iter=LASSO_MAX_ITER,
        tol=LASSO_TOL,
        warm_start=True, # warm_start=True is important. The alpha values are sorted from largest to smallest. The solution for a large alpha is a good starting point for the next slightly smaller alpha. Warm starts let the model reuse the previous fitted coefficients instead of starting from scratch each time. This can make the alpha path much faster.
        random_state=9890
    )
    
    for alpha in alphas: # the function loops through the alpha values
        model.alpha = alpha
        
        with warnings.catch_warnings(record=True) as caught_warnings:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(X_train_scaled, y_train)              # For each alpha, it fits the model, predicts on the training set and validation set, and records the MSE.
            
            convergence_warning = any(
                issubclass(w.category, ConvergenceWarning)
                for w in caught_warnings
            )   # A convergence warning means sklearn is saying, roughly, “I stopped before fully solving the optimization problem.” That does not always make the result useless, but it is a warning that you may need more iterations, stronger regularization, better scaling, or a different tolerance.
        
        train_pred = model.predict(X_train_scaled)
        val_pred = model.predict(X_val_scaled)
        
        nonzero_coef = int(np.sum(np.abs(model.coef_) > 1e-8))  # counts how many coefficients are nonzero:
        
        rows.append({
            "model_class": "Lasso",
            "stage": stage,
            "feature_space": feature_space_name,
            "scaler": scaler_name,
            "alpha": alpha,
            "train_mse": mean_squared_error(y_train, train_pred),  # tells you how well the model fits the training split.
            "val_mse": mean_squared_error(y_val, val_pred),  # is the main number used for model selection.
            "nonzero_coef": nonzero_coef,  #  tells you how sparse the model is.
            "n_iter": model.n_iter_,  # tells you how many iterations the solver used.
            "convergence_warning": convergence_warning,
            "aic": np.nan,
            "bic": np.nan
        })
    
    return pd.DataFrame(rows)

# ------------------------------------------------------------
# Stage 1: broad search
# ------------------------------------------------------------

start_time = time.perf_counter()

lasso_broad_rows = []
alpha_max_records = []
# This loops over every combination of feature space and scaler.There are four feature spaces and two scalers, so there are eight combinations:
for feature_space_name, (X_train_space, X_val_space) in feature_spaces.items():
    for scaler_name, scaler_factory in scaler_factories.items():
        
        scaler = scaler_factory()
        X_train_scaled = scaler.fit_transform(X_train_space)
        
        alpha_max = compute_lasso_alpha_max(X_train_scaled, y_tr)  # For each combination, it computes that setup’s alpha_max
        alpha_min = alpha_max * LASSO_MIN_ALPHA_RATIO
        
        # Descending alpha path for warm starts
        broad_alphas = np.logspace(
            np.log10(alpha_max),
            np.log10(alpha_min),
            LASSO_BROAD_GRID_SIZE
        ) # Because the first argument is larger than the second, this creates descending values. That is intentional. The model starts with the strongest penalty and moves toward weaker penalties. This works well with warm_start=True.
        
        alpha_max_records.append({ # The code records the alpha range
            "feature_space": feature_space_name,
            "scaler": scaler_name,
            "alpha_max": alpha_max,
            "alpha_min": alpha_min
        })
        
        result = fit_lasso_path( # Then it fits all 25 broad-search Lasso models for that setup:
            X_train_space=X_train_space,
            X_val_space=X_val_space,
            y_train=y_tr,
            y_val=y_val,
            alphas=broad_alphas,
            scaler_name=scaler_name,
            scaler_factory=scaler_factory,
            feature_space_name=feature_space_name,
            stage="broad"
        )
        
        lasso_broad_rows.append(result)
# After the loops finish, it combines all broad-search results:
lasso_alpha_max_table = pd.DataFrame(alpha_max_records)
lasso_broad_results = pd.concat(lasso_broad_rows, ignore_index=True) # Since there are eight setup combinations and 25 alphas each, the broad search fits 8 × 25 = 200 Lasso models

best_broad_by_space_scaler = ( # picks the best broad alpha for each feature-space/scaler pair:
    lasso_broad_results
    .loc[lasso_broad_results.groupby(["feature_space", "scaler"])["val_mse"].idxmin()]
    .sort_values("val_mse")
)   # This groups the results by feature_space and scaler, finds the row with the smallest validation MSE in each group, and sorts those eight winners from best to worst.

print("Lasso alpha_max table:")
print(lasso_alpha_max_table.to_string(index=False))

print("\nBest broad Lasso result by feature space and scaler:")
print(best_broad_by_space_scaler.to_string(index=False))


# ------------------------------------------------------------
# Stage 2: fine search around each broad-search winner
# ------------------------------------------------------------

lasso_fine_rows = []

for _, row in best_broad_by_space_scaler.iterrows():   # The code loops over the eight broad-search winners
    feature_space_name = row["feature_space"]
    scaler_name = row["scaler"]
    scaler_factory = scaler_factories[scaler_name]
    best_alpha = row["alpha"]   # For each winner, it gets the best alpha from the broad search:
    
    alpha_record = lasso_alpha_max_table[ # Then it retrieves the original alpha bounds for that feature-space/scaler combination:
        (lasso_alpha_max_table["feature_space"] == feature_space_name) &
        (lasso_alpha_max_table["scaler"] == scaler_name)
    ].iloc[0]
    
    alpha_max = alpha_record["alpha_max"]
    alpha_min = alpha_record["alpha_min"]
    # Then it builds a narrower alpha range around the broad-search winner:
    best_log = np.log10(best_alpha)
    fine_low = max(np.log10(alpha_min), best_log - LASSO_FINE_WIDTH_LOG10)
    fine_high = min(np.log10(alpha_max), best_log + LASSO_FINE_WIDTH_LOG10) # Because LASSO_FINE_WIDTH_LOG10 = 0.5, this searches approximately 3.16 times below and 3.16 times above the broad-search winner, clipped so it never goes outside the original broad-search range.
    
    # Descending alpha path for warm starts
    fine_alphas = np.logspace(fine_high, fine_low, LASSO_FINE_GRID_SIZE)  # this creates descending alpha values for warm starts. The fine search fits 21 more models for each of the eight broad winners: 8 × 21 = 168 Lasso models. so the whole cell fits 200 broad models + 168 fine models = 368 Lasso models
    
    X_train_space, X_val_space = feature_spaces[feature_space_name]
    
    result = fit_lasso_path(
        X_train_space=X_train_space,
        X_val_space=X_val_space,
        y_train=y_tr,
        y_val=y_val,
        alphas=fine_alphas,
        scaler_name=scaler_name,
        scaler_factory=scaler_factory,
        feature_space_name=feature_space_name,
        stage="fine"
    )
    
    lasso_fine_rows.append(result)

lasso_fine_results = pd.concat(lasso_fine_rows, ignore_index=True)

# ------------------------------------------------------------
# Combine broad + fine results and select winners
# ------------------------------------------------------------

lasso_results = pd.concat(
    [lasso_broad_results, lasso_fine_results],
    ignore_index=True
)

best_lasso_by_space = ( # Then it finds the best Lasso model for each feature space. This gives one winner for each of the four feature spaces, regardless of scaler, alpha, or search stage.
    lasso_results
    .loc[lasso_results.groupby("feature_space")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

best_lasso_overall = lasso_results.loc[lasso_results["val_mse"].idxmin()]  # Then it finds the single best Lasso model overall

elapsed = time.perf_counter() - start_time

print("\nBest Lasso result by feature space:")
print(best_lasso_by_space.to_string(index=False))

print("\nOverall best Lasso result:")
print(best_lasso_overall)

print(f"\nElapsed time: {elapsed:.2f} seconds")

print("\nConvergence warning counts:")
print(
    lasso_results
    .groupby(["feature_space", "scaler", "stage"])["convergence_warning"]
    .sum()
    .reset_index()
    .to_string(index=False)
)

Lasso alpha_max table:
              feature_space   scaler  alpha_max  alpha_min
                       base standard  11.255885   0.001126
                       base   robust   9.898994   0.000990
             base_plus_poly standard  11.255885   0.001126
             base_plus_poly   robust  23.219510   0.002322
     base_plus_interactions standard  11.255885   0.001126
     base_plus_interactions   robust   9.898994   0.000990
base_plus_poly_interactions standard  11.255885   0.001126
base_plus_poly_interactions   robust  23.219510   0.002322

Best broad Lasso result by feature space and scaler:
model_class stage               feature_space   scaler    alpha  train_mse    val_mse  nonzero_coef  n_iter  convergence_warning  aic  bic
      Lasso broad base_plus_poly_interactions standard 0.001126 301.032379 308.416448           133   30000                 True  NaN  NaN
      Lasso broad base_plus_poly_interactions   robust 0.002322 301.362320 308.808215           128    6321       

### Lasso Regression Results

Lasso regression was evaluated across the four controlled feature spaces:

1. Base feature space
2. Base + polynomial terms
3. Base + interaction terms
4. Base + polynomial + interaction terms

Both `StandardScaler` and `RobustScaler` were compared. Alpha was tuned using a data-driven two-stage search based on `alpha_max`.

Best Lasso model:
- Feature space: base + polynomial + interactions
- Scaler: StandardScaler
- Alpha: 0.001126
- Train MSE: 301.03
- Validation MSE: 308.42
- Nonzero coefficients: 134

Interpretation:
- The best Lasso model used the combined polynomial + interaction feature space.
- StandardScaler outperformed RobustScaler.
- The best alpha was at the lowest value in the search grid, indicating that weak regularization performed best.
- Lasso retained 134 nonzero coefficients, so it did not produce a highly sparse model.
- This supports the earlier finding that predictive signal is distributed across many predictors rather than concentrated in a small subset.

Conclusion:
Lasso provides useful feature selection but does not currently outperform Ridge or backward stepwise selection. The results suggest that aggressive sparsity is not ideal for this dataset.

### Process above: 

Try Lasso on every engineered feature set, try both standard and robust scaling, choose a sensible alpha range based on the data, do a broad alpha search, refine around the best alpha, track validation MSE and sparsity, then report the best Lasso configuration to compare against Ridge and earlier linear models.

In [26]:
# ============================================================
# 11A. Elastic Net Regression: two-stage search
# ============================================================

from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

# ------------------------------------------------------------
# Design choices
# ------------------------------------------------------------
# - Feature spaces: all four controlled feature spaces
# - Scalers: StandardScaler and RobustScaler
# - l1_ratio grid: from mostly-Ridge to near-Lasso
# - alpha grid: data-driven alpha_max, then two-stage search
# - Validation: current development holdout split
# - CV: intentionally postponed until finalist screening
# - Checkpointing: saves results after every path

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

ELASTICNET_RESULTS_PATH = RESULTS_DIR / "elasticnet_results_holdout.csv"
ELASTICNET_BEST_BY_SPACE_PATH = RESULTS_DIR / "elasticnet_best_by_space_holdout.csv"
ELASTICNET_BEST_OVERALL_PATH = RESULTS_DIR / "elasticnet_best_overall_holdout.csv"

OVERWRITE_ELASTICNET_RESULTS = True

if OVERWRITE_ELASTICNET_RESULTS:
    for path in [
        ELASTICNET_RESULTS_PATH,
        ELASTICNET_BEST_BY_SPACE_PATH,
        ELASTICNET_BEST_OVERALL_PATH,
    ]:
        if path.exists():
            path.unlink()

scaler_factories = {
    "standard": StandardScaler,
    "robust": RobustScaler
}

elasticnet_l1_ratios = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

ELASTICNET_BROAD_GRID_SIZE = 25
ELASTICNET_FINE_GRID_SIZE = 21
ELASTICNET_MIN_ALPHA_RATIO = 1e-4
ELASTICNET_FINE_WIDTH_LOG10 = 0.5

ELASTICNET_MAX_ITER = 50000
ELASTICNET_TOL = 1e-4


def append_checkpoint(df, path):
    """Append results to a CSV checkpoint file."""
    write_header = not path.exists()
    df.to_csv(path, mode="a", header=write_header, index=False)


def compute_alpha_max_lasso_base(X_scaled, y):
    """
    Computes max_j |x_j^T (y - ybar)| / n.

    For Elastic Net, alpha_max depends on l1_ratio:
        alpha_max_enet = alpha_max_lasso_base / l1_ratio

    This matches sklearn's ElasticNet objective:
        (1 / (2n)) * ||y - Xw||^2
        + alpha * l1_ratio * ||w||_1
        + 0.5 * alpha * (1 - l1_ratio) * ||w||_2^2
    """
    y_arr = np.asarray(y, dtype=float)
    y_centered = y_arr - y_arr.mean()
    n = X_scaled.shape[0]
    alpha_max_base = np.max(np.abs(X_scaled.T @ y_centered)) / n
    
    if alpha_max_base <= 0:
        raise ValueError("alpha_max_base is non-positive; check X/y inputs.")
    
    return float(alpha_max_base)


def fit_elasticnet_path_scaled(
    X_train_scaled,
    X_val_scaled,
    y_train,
    y_val,
    alphas,
    l1_ratio,
    feature_space_name,
    scaler_name,
    stage
):
    """
    Fits Elastic Net along a descending alpha path using warm starts.
    """
    rows = []
    
    alphas = np.asarray(alphas, dtype=float)
    
    model = ElasticNet(
        alpha=alphas[0],
        l1_ratio=l1_ratio,
        fit_intercept=True,
        max_iter=ELASTICNET_MAX_ITER,
        tol=ELASTICNET_TOL,
        warm_start=True,
        selection="cyclic",
        random_state=9890
    )
    
    for alpha in alphas:
        model.alpha = float(alpha)
        
        with warnings.catch_warnings(record=True) as caught_warnings:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(X_train_scaled, y_train)
            
            convergence_warning = any(
                issubclass(w.category, ConvergenceWarning)
                for w in caught_warnings
            )
        
        train_pred = model.predict(X_train_scaled)
        val_pred = model.predict(X_val_scaled)
        
        rows.append({
            "model_class": "ElasticNet",
            "stage": stage,
            "feature_space": feature_space_name,
            "scaler": scaler_name,
            "l1_ratio": l1_ratio,
            "alpha": float(alpha),
            "train_mse": mean_squared_error(y_train, train_pred),
            "val_mse": mean_squared_error(y_val, val_pred),
            "nonzero_coef": int(np.sum(np.abs(model.coef_) > 1e-8)),
            "n_iter": int(model.n_iter_),
            "convergence_warning": convergence_warning,
            "aic": np.nan,
            "bic": np.nan
        })
    
    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Main search
# ------------------------------------------------------------

start_time = time.perf_counter()
elasticnet_all_parts = []

print("Elastic Net overnight search starting...")
print("Feature spaces:", list(feature_spaces.keys()))
print("l1_ratios:", elasticnet_l1_ratios)

for feature_space_name, (X_train_space, X_val_space) in feature_spaces.items():
    for scaler_name, scaler_factory in scaler_factories.items():
        
        print("\n" + "=" * 90)
        print(f"Feature space: {feature_space_name} | Scaler: {scaler_name}")
        print("=" * 90)
        
        scaler = scaler_factory()
        X_train_scaled = scaler.fit_transform(X_train_space)
        X_val_scaled = scaler.transform(X_val_space)
        
        alpha_max_base = compute_alpha_max_lasso_base(X_train_scaled, y_tr)
        
        broad_parts_this_combo = []
        
        # ----------------------------------------------------
        # Stage 1: broad search for each l1_ratio
        # ----------------------------------------------------
        for l1_ratio in elasticnet_l1_ratios:
            alpha_max = alpha_max_base / l1_ratio
            alpha_min = alpha_max * ELASTICNET_MIN_ALPHA_RATIO
            
            broad_alphas = np.logspace(
                np.log10(alpha_max),
                np.log10(alpha_min),
                ELASTICNET_BROAD_GRID_SIZE
            )
            
            broad_df = fit_elasticnet_path_scaled(
                X_train_scaled=X_train_scaled,
                X_val_scaled=X_val_scaled,
                y_train=y_tr,
                y_val=y_val,
                alphas=broad_alphas,
                l1_ratio=l1_ratio,
                feature_space_name=feature_space_name,
                scaler_name=scaler_name,
                stage="broad"
            )
            
            broad_parts_this_combo.append(broad_df)
            elasticnet_all_parts.append(broad_df)
            append_checkpoint(broad_df, ELASTICNET_RESULTS_PATH)
        
        broad_combo_df = pd.concat(broad_parts_this_combo, ignore_index=True)
        
        best_broad_by_l1 = (
            broad_combo_df
            .loc[broad_combo_df.groupby("l1_ratio")["val_mse"].idxmin()]
            .sort_values("val_mse")
        )
        
        print("\nBest broad result by l1_ratio:")
        print(
            best_broad_by_l1[
                ["l1_ratio", "alpha", "train_mse", "val_mse", "nonzero_coef", "convergence_warning"]
            ].to_string(index=False)
        )
        
        # ----------------------------------------------------
        # Stage 2: fine search around each broad winner
        # ----------------------------------------------------
        for _, row in best_broad_by_l1.iterrows():
            l1_ratio = float(row["l1_ratio"])
            best_alpha = float(row["alpha"])
            
            alpha_max = alpha_max_base / l1_ratio
            alpha_min = alpha_max * ELASTICNET_MIN_ALPHA_RATIO
            
            best_log = np.log10(best_alpha)
            fine_low = max(np.log10(alpha_min), best_log - ELASTICNET_FINE_WIDTH_LOG10)
            fine_high = min(np.log10(alpha_max), best_log + ELASTICNET_FINE_WIDTH_LOG10)
            
            fine_alphas = np.logspace(
                fine_high,
                fine_low,
                ELASTICNET_FINE_GRID_SIZE
            )
            
            fine_df = fit_elasticnet_path_scaled(
                X_train_scaled=X_train_scaled,
                X_val_scaled=X_val_scaled,
                y_train=y_tr,
                y_val=y_val,
                alphas=fine_alphas,
                l1_ratio=l1_ratio,
                feature_space_name=feature_space_name,
                scaler_name=scaler_name,
                stage="fine"
            )
            
            elasticnet_all_parts.append(fine_df)
            append_checkpoint(fine_df, ELASTICNET_RESULTS_PATH)
        
        elapsed_so_far = time.perf_counter() - start_time
        print(f"\nFinished {feature_space_name} | {scaler_name}. Elapsed seconds: {elapsed_so_far:.2f}")


# ------------------------------------------------------------
# Summarize final Elastic Net results
# ------------------------------------------------------------

elasticnet_results = pd.concat(elasticnet_all_parts, ignore_index=True)

best_elasticnet_by_space = (
    elasticnet_results
    .loc[elasticnet_results.groupby("feature_space")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

best_elasticnet_overall = elasticnet_results.loc[
    elasticnet_results["val_mse"].idxmin()
]

best_elasticnet_by_space.to_csv(ELASTICNET_BEST_BY_SPACE_PATH, index=False)
best_elasticnet_overall.to_frame().T.to_csv(ELASTICNET_BEST_OVERALL_PATH, index=False)

elapsed = time.perf_counter() - start_time

print("\n" + "=" * 90)
print("Best Elastic Net result by feature space:")
print("=" * 90)
print(best_elasticnet_by_space.to_string(index=False))

print("\n" + "=" * 90)
print("Overall best Elastic Net result:")
print("=" * 90)
print(best_elasticnet_overall)

print(f"\nElapsed time: {elapsed:.2f} seconds")

print("\nConvergence warning counts:")
print(
    elasticnet_results
    .groupby(["feature_space", "scaler", "stage"])["convergence_warning"]
    .sum()
    .reset_index()
    .to_string(index=False)
)

print("\nSaved files:")
print("All Elastic Net results:", ELASTICNET_RESULTS_PATH)
print("Best by feature space:", ELASTICNET_BEST_BY_SPACE_PATH)
print("Best overall:", ELASTICNET_BEST_OVERALL_PATH)

Elastic Net overnight search starting...
Feature spaces: ['base', 'base_plus_poly', 'base_plus_interactions', 'base_plus_poly_interactions']
l1_ratios: [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]

Feature space: base | Scaler: standard

Best broad result by l1_ratio:
 l1_ratio    alpha  train_mse    val_mse  nonzero_coef  convergence_warning
     0.99 0.001137 305.154413 312.619191           132                False
     0.95 0.001185 305.158777 312.623589           135                False
     0.90 0.001251 305.164429 312.629943           145                False
     0.75 0.001501 305.192217 312.655747           154                False
     0.50 0.002251 305.267042 312.737288           156                False
     0.25 0.004502 305.469335 312.955788           155                False
     0.10 0.011256 305.948373 313.426435           156                False
     0.05 0.022512 306.670214 314.083149           162                False

Finished base | standard. Elapsed seconds: 13

### Elastic Net Regression Results

Elastic Net regression was evaluated across the four controlled feature spaces:

1. Base feature space
2. Base + polynomial terms
3. Base + interaction terms
4. Base + polynomial + interaction terms

Both `StandardScaler` and `RobustScaler` were compared, and the model was tuned over multiple `l1_ratio` values and data-driven alpha values.

Best Elastic Net model:
- Feature space: base + polynomial + interactions
- Scaler: StandardScaler
- l1_ratio: 0.99
- Alpha: 0.001137
- Train MSE: 301.05
- Validation MSE: 308.43
- Nonzero coefficients: 152

Interpretation:
- Elastic Net performed best on the combined polynomial + interaction feature space.
- The best `l1_ratio` was 0.99, meaning the model behaved very similarly to Lasso.
- StandardScaler outperformed RobustScaler.
- Elastic Net retained 152 nonzero coefficients, so it did not produce a highly sparse model.
- The result is competitive but does not outperform Ridge or the best backward stepwise model.

Conclusion:
Elastic Net confirms that the strongest feature space is the combined polynomial + interaction space, but it does not improve over the current best model. The current best linear-family model remains backward stepwise on the combined feature space, with validation MSE around 308.10.

In [28]:
# ============================================================
# 12A. Step Function Models
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import time

# ------------------------------------------------------------
# Design:
# - Select top continuous predictors from simple LR ranking
# - Create step-function dummy variables using training-split cutpoints only
# - Add step features to each existing controlled feature space
# - Evaluate using current development holdout split
# ------------------------------------------------------------

start_time = time.perf_counter()

# Select top continuous features only.
# Exclude binary dummies / missing indicators by requiring many unique values.
step_candidate_features = []

for col in simple_lr_results["feature"].tolist():
    if col in X_tr.columns:
        unique_count = X_tr[col].nunique(dropna=False)
        if unique_count >= 20 and not col.endswith("_missing"):
            step_candidate_features.append(col)
    
    if len(step_candidate_features) >= 10:
        break

print("Step-function candidate features:")
for i, col in enumerate(step_candidate_features, start=1):
    print(f"{i:2d}. {col}")


def make_step_features(X_train_source, X_val_source, columns, n_bins, method):
    """
    Build step-function dummy features.

    Cutpoints are learned from X_train_source only.
    Then the same cutpoints are applied to X_val_source.

    method:
        'quantile'    -> bins based on training quantiles
        'equal_width' -> bins based on training min/max width
    """
    train_parts = []
    val_parts = []
    actual_step_cols = []

    for col in columns:
        x_train = X_train_source[col].astype(float)
        x_val = X_val_source[col].astype(float)

        if x_train.nunique(dropna=False) < 2:
            continue

        if method == "quantile":
            try:
                _, edges = pd.qcut(
                    x_train,
                    q=n_bins,
                    retbins=True,
                    duplicates="drop"
                )
            except ValueError:
                continue

        elif method == "equal_width":
            min_val = x_train.min()
            max_val = x_train.max()

            if not np.isfinite(min_val) or not np.isfinite(max_val) or min_val == max_val:
                continue

            edges = np.linspace(min_val, max_val, n_bins + 1)

        else:
            raise ValueError("method must be 'quantile' or 'equal_width'")

        edges = np.unique(edges)

        if len(edges) <= 2:
            continue

        # Make validation robust to values slightly outside training range.
        edges = edges.astype(float)
        edges[0] = -np.inf
        edges[-1] = np.inf

        k = len(edges) - 1

        train_codes = pd.cut(
            x_train,
            bins=edges,
            labels=False,
            include_lowest=True
        )

        val_codes = pd.cut(
            x_val,
            bins=edges,
            labels=False,
            include_lowest=True
        )

        train_cat = pd.Categorical(train_codes, categories=list(range(k)))
        val_cat = pd.Categorical(val_codes, categories=list(range(k)))

        prefix = f"{col}_step_{method}_{n_bins}"

        train_dummies = pd.get_dummies(
            train_cat,
            prefix=prefix,
            drop_first=True,
            dtype=float
        )
        val_dummies = pd.get_dummies(
            val_cat,
            prefix=prefix,
            drop_first=True,
            dtype=float
        )

        train_dummies.index = X_train_source.index
        val_dummies.index = X_val_source.index

        train_dummies, val_dummies = train_dummies.align(
            val_dummies,
            join="left",
            axis=1,
            fill_value=0
        )

        train_parts.append(train_dummies)
        val_parts.append(val_dummies)
        actual_step_cols.extend(train_dummies.columns.tolist())

    if len(train_parts) == 0:
        empty_train = pd.DataFrame(index=X_train_source.index)
        empty_val = pd.DataFrame(index=X_val_source.index)
        return empty_train, empty_val, []

    X_train_steps = pd.concat(train_parts, axis=1)
    X_val_steps = pd.concat(val_parts, axis=1)

    return X_train_steps, X_val_steps, actual_step_cols


step_methods = ["quantile", "equal_width"]
step_bins_grid = [3, 5, 10]

step_results_rows = []

for base_space_name, (X_train_space, X_val_space) in feature_spaces.items():
    for method in step_methods:
        for n_bins in step_bins_grid:

            X_train_steps, X_val_steps, step_cols = make_step_features(
                X_train_source=X_tr,
                X_val_source=X_val,
                columns=step_candidate_features,
                n_bins=n_bins,
                method=method
            )

            X_train_aug = pd.concat([X_train_space, X_train_steps], axis=1)
            X_val_aug = pd.concat([X_val_space, X_val_steps], axis=1)

            model = LinearRegression()
            model.fit(X_train_aug, y_tr)

            train_pred = model.predict(X_train_aug)
            val_pred = model.predict(X_val_aug)

            step_results_rows.append({
                "model_class": "Step Functions + Linear Regression",
                "base_space": base_space_name,
                "step_method": method,
                "n_bins": n_bins,
                "n_step_features_added": len(step_cols),
                "total_features": X_train_aug.shape[1],
                "train_mse": mean_squared_error(y_tr, train_pred),
                "val_mse": mean_squared_error(y_val, val_pred),
                "aic": np.nan,
                "bic": np.nan
            })

step_results = pd.DataFrame(step_results_rows)

best_step_by_base_space = (
    step_results
    .loc[step_results.groupby("base_space")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

best_step_overall = step_results.loc[step_results["val_mse"].idxmin()]

elapsed = time.perf_counter() - start_time

print("\nBest step-function model by base space:")
print(best_step_by_base_space.to_string(index=False))

print("\nOverall best step-function model:")
print(best_step_overall)

print(f"\nElapsed time: {elapsed:.2f} seconds")

Step-function candidate features:
 1. PERCENT_ECONOMICALLY_DISADVANTAGED
 2. PERCENT_FREE_LUNCH
 3. PERCENT_DIPLOMA
 4. PERCENT_STILL_ENROLLED
 5. PERCENT_HOMELESS
 6. PERCENT_BLACK
 7. PERCENT_WITH_DISABILITIES
 8. PERCENT_ENGLISH_LANGUAGE_LEANERS
 9. ATTENDANCE_RATE
10. PERCENT_WHITE

Best step-function model by base space:
                       model_class                  base_space step_method  n_bins  n_step_features_added  total_features  train_mse    val_mse  aic  bic
Step Functions + Linear Regression base_plus_poly_interactions    quantile      10                     83             265 296.544041 303.570375  NaN  NaN
Step Functions + Linear Regression              base_plus_poly    quantile      10                     83             255 297.288261 304.119794  NaN  NaN
Step Functions + Linear Regression      base_plus_interactions    quantile      10                     83             255 297.540321 304.612830  NaN  NaN
Step Functions + Linear Regression                      

### Step Function Model Results

Step-function models were evaluated by binning the strongest continuous predictors from the simple linear regression ranking.

Candidate variables included:
- `PERCENT_ECONOMICALLY_DISADVANTAGED`
- `PERCENT_FREE_LUNCH`
- `PERCENT_DIPLOMA`
- `PERCENT_STILL_ENROLLED`
- `PERCENT_HOMELESS`
- `PERCENT_BLACK`
- `PERCENT_WITH_DISABILITIES`
- `PERCENT_ENGLISH_LANGUAGE_LEANERS`
- `ATTENDANCE_RATE`
- `PERCENT_WHITE`

The best step-function model used:
- Base feature space: base + polynomial + interactions
- Binning method: quantile bins
- Number of bins: 10
- Step-function features added: 83
- Total features: 265

Results:
- Train MSE: 296.54
- Validation MSE: 303.57

Interpretation:
- Step functions substantially improved performance compared with previous linear-family models.
- The improvement suggests that some important predictors have threshold-based or piecewise relationships with the target.
- Quantile binning performed best, likely because it creates balanced bins across skewed continuous predictors.
- The train-validation gap remains moderate, so the improvement does not appear to be caused by severe overfitting.

Conclusion:
Step-function feature engineering is the strongest modeling direction so far. The current best model is a linear regression model using the combined polynomial + interaction feature space plus quantile step functions.

In [29]:
# ============================================================
# 12B. Expanded Step Function Search
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import time

start_time = time.perf_counter()

# ------------------------------------------------------------
# Helper: get top continuous features from simple LR ranking
# ------------------------------------------------------------

def get_top_continuous_features(k, min_unique=20):
    features = []
    
    for col in simple_lr_results["feature"].tolist():
        if col not in X_tr.columns:
            continue
        
        unique_count = X_tr[col].nunique(dropna=False)
        
        if unique_count >= min_unique and not col.endswith("_missing"):
            features.append(col)
        
        if len(features) >= k:
            break
    
    return features


# ------------------------------------------------------------
# Expanded search design
# ------------------------------------------------------------

candidate_feature_counts = [10, 15, 20]
step_bins_grid = [5, 10, 15, 20]
step_method = "quantile"

expanded_step_rows = []

for k in candidate_feature_counts:
    candidate_features = get_top_continuous_features(k)
    
    print("\n" + "=" * 80)
    print(f"Top {k} continuous step-function candidates:")
    print(candidate_features)
    
    for base_space_name, (X_train_space, X_val_space) in feature_spaces.items():
        for n_bins in step_bins_grid:
            
            X_train_steps, X_val_steps, step_cols = make_step_features(
                X_train_source=X_tr,
                X_val_source=X_val,
                columns=candidate_features,
                n_bins=n_bins,
                method=step_method
            )
            
            X_train_aug = pd.concat([X_train_space, X_train_steps], axis=1)
            X_val_aug = pd.concat([X_val_space, X_val_steps], axis=1)
            
            model = LinearRegression()
            model.fit(X_train_aug, y_tr)
            
            train_pred = model.predict(X_train_aug)
            val_pred = model.predict(X_val_aug)
            
            expanded_step_rows.append({
                "model_class": "Expanded Step Functions + Linear Regression",
                "base_space": base_space_name,
                "step_method": step_method,
                "candidate_feature_count": k,
                "n_bins": n_bins,
                "n_step_features_added": len(step_cols),
                "total_features": X_train_aug.shape[1],
                "train_mse": mean_squared_error(y_tr, train_pred),
                "val_mse": mean_squared_error(y_val, val_pred),
                "aic": np.nan,
                "bic": np.nan
            })

expanded_step_results = pd.DataFrame(expanded_step_rows)

best_expanded_step_by_base_space = (
    expanded_step_results
    .loc[expanded_step_results.groupby("base_space")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

best_expanded_step_overall = expanded_step_results.loc[
    expanded_step_results["val_mse"].idxmin()
]

elapsed = time.perf_counter() - start_time

print("\nBest expanded step-function model by base space:")
print(best_expanded_step_by_base_space.to_string(index=False))

print("\nOverall best expanded step-function model:")
print(best_expanded_step_overall)

print(f"\nElapsed time: {elapsed:.2f} seconds")


Top 10 continuous step-function candidates:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE']

Top 15 continuous step-function candidates:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE', 'PERCENT_DROPOUT', 'PERCENT_HISPANIC', 'PERCENT_ASIAN', 'GRADE_12', 'GRADE_11']

Top 20 continuous step-function candidates:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE', 'PERCENT_DROPOUT', 'PERCENT_HISPANIC', 'PERCENT_ASIAN

### Expanded Step Function Search

The expanded step-function search tested quantile-based binning over larger sets of continuous predictors.

Best model:
- Base feature space: base + polynomial + interactions
- Candidate continuous predictors: 20
- Quantile bins: 20
- Step-function features added: 252
- Total features: 434

Results:
- Train MSE: 289.14
- Validation MSE: 296.14

Interpretation:
- Step functions substantially improved validation performance compared with all previous linear-family models.
- The improvement suggests that several predictors have threshold-based or piecewise relationships with `PERCENT_PROFICIENT`.
- The best model occurred at the largest tested feature count and bin count, indicating that the useful search space may not yet be exhausted.
- The train-validation gap remains moderate, so the model does not appear to be severely overfit at this stage.

Conclusion:
Step-function feature engineering is currently the strongest modeling direction. The best model so far is a linear regression model using the combined polynomial + interaction feature space plus quantile step functions.

In [30]:
# ============================================================
# 12C. Focused expanded step-function search on best base space
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import time

start_time = time.perf_counter()

# Current best base space
best_base_space_name = "base_plus_poly_interactions"
X_train_best_base, X_val_best_base = feature_spaces[best_base_space_name]

# Expand beyond previous boundary.
candidate_feature_counts = [20, 25, 30]
step_bins_grid = [20, 25, 30]
step_method = "quantile"

focused_step_rows = []

for k in candidate_feature_counts:
    candidate_features = get_top_continuous_features(k)
    
    print("\n" + "=" * 80)
    print(f"Top {k} continuous step-function candidates:")
    print(candidate_features)
    
    for n_bins in step_bins_grid:
        X_train_steps, X_val_steps, step_cols = make_step_features(
            X_train_source=X_tr,
            X_val_source=X_val,
            columns=candidate_features,
            n_bins=n_bins,
            method=step_method
        )
        
        X_train_aug = pd.concat([X_train_best_base, X_train_steps], axis=1)
        X_val_aug = pd.concat([X_val_best_base, X_val_steps], axis=1)
        
        model = LinearRegression()
        model.fit(X_train_aug, y_tr)
        
        train_pred = model.predict(X_train_aug)
        val_pred = model.predict(X_val_aug)
        
        focused_step_rows.append({
            "model_class": "Focused Expanded Step Functions + Linear Regression",
            "base_space": best_base_space_name,
            "step_method": step_method,
            "candidate_feature_count": k,
            "n_bins": n_bins,
            "n_step_features_added": len(step_cols),
            "total_features": X_train_aug.shape[1],
            "train_mse": mean_squared_error(y_tr, train_pred),
            "val_mse": mean_squared_error(y_val, val_pred),
            "aic": np.nan,
            "bic": np.nan
        })

focused_step_results = pd.DataFrame(focused_step_rows).sort_values("val_mse")

best_focused_step_overall = focused_step_results.loc[
    focused_step_results["val_mse"].idxmin()
]

elapsed = time.perf_counter() - start_time

print("\nFocused expanded step-function results:")
print(focused_step_results.to_string(index=False))

print("\nOverall best focused expanded step-function model:")
print(best_focused_step_overall)

print(f"\nElapsed time: {elapsed:.2f} seconds")


Top 20 continuous step-function candidates:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE', 'PERCENT_DROPOUT', 'PERCENT_HISPANIC', 'PERCENT_ASIAN', 'GRADE_12', 'GRADE_11', 'PRE_K', 'GRADE_10', 'NUMBER_OF_TEACHERS', 'GRADE_09', 'NUMBER_OF_COUNSELORS']

Top 25 continuous step-function candidates:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE', 'PERCENT_DROPOUT', 'PERCENT_HISPANIC', 'PERCENT_ASIAN', 'GRADE_12', 'GRADE_11', 'PRE_K', 'GRADE_10', 'NUMBER_OF_TEACHERS', 'GRADE_09', 'NUMBER_OF_COUNSELORS', 'N_PUPILS', 'FEDERAL_FUNDING_PER_PUPIL', 'GRADE_04', 'GRADE_03', 'SCHOOL_freq']

Top 30 continuous step-